In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

from aiba import optimize_intervals

In [ ]:
# Data loading and preprocessing
column_names = [
    "Center of buoyancy", "Prismatic coefficient", "Length-displacement",
    "Beam-draught", "Length-beam", "Froude number", "Residuary resistance"
]
df = pd.read_csv("yacht_hydrodynamics.txt", sep=r";", header=None, names=column_names)
X = df.drop(columns=["Residuary resistance"])
y = df["Residuary resistance"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=0)

In [ ]:
# Funciones de muestreo
def sample_log(low, high):
    return float(np.exp(np.random.uniform(np.log(low), np.log(high))))

def sample_uniform(low, high):
    return float(np.random.uniform(low, high))

# Generación de muestras del espacio de hiperparámetros

Con `use_sample_data = True` se cargan evaluaciones pregeneradas (`yacht_model_configs_test.csv`); con `False` se vuelven a entrenar los modelos. AIBA recibe únicamente los siete hiperparámetros controlables. `actual_epochs`, `num_params` y `Quality` son resultados posteriores al entrenamiento y se excluyen del espacio de búsqueda. Como AIBA minimiza y `Quality` es R², el objetivo utilizado es `-Quality`.

In [ ]:
# Generar o cargar evaluaciones del espacio amplio de hiperparámetros
configs = []
scores = []

input_dim = X.shape[1]
output_dim = 1

sampling_type = {
    "batch_size": "log",
    "max_epochs": "log",
    "learning_rate": "log",
    "neurons": "log",
    "dropout_rate": "log",
    "num_hidden_layers": "uniform",
    "activation": "uniform"
}

activation_mapping = {
        'relu': 0,
        'sigmoid': 0.5,
        'tanh': 1
    }

HYPERPARAMETER_COLUMNS = [
    "batch_size", "max_epochs", "learning_rate", "neurons",
    "dropout_rate", "num_hidden_layers", "activation"
]

use_sample_data = True  # Cambiar a False para usar todo el dataset

if not use_sample_data:
    num_models = 1000
    for i in range(num_models):

        print(f"Training model {i+1}/{num_models}, average score so far: {np.mean(scores) if scores else 0:.4f}", end="\r")

        # Hiperparámetros
        batch_size = int(sample_log(1, 256))
        max_epochs = int(sample_log(1, 400))
        learning_rate = sample_log(1e-5, 1e-1)
        neurons = int(sample_log(3, 600))
        dropout_rate = sample_log(1e-5, 0.7)
        num_hidden_layers = int(sample_uniform(1, 4))
        activation = sample_uniform(0, 2)/2  # Placeholder, no se usa en MLPRegressor
        activation_str = min(activation_mapping.keys(), key=lambda k: abs(activation_mapping[k] - activation))

        # Arquitectura dinámica
        model = keras.Sequential()
        model.add(keras.layers.Input(shape=(input_dim,)))
        for _ in range(num_hidden_layers):
            model.add(keras.layers.Dense(neurons, activation=activation_str))
            model.add(keras.layers.Dropout(dropout_rate))
        model.add(keras.layers.Dense(output_dim))  # salida lineal

        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss="mse")

        # Dividir dataset
        X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=i)

        # Entrenamiento
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=max_epochs,
            batch_size=batch_size,
            verbose=0
        )

        # Evaluación
        preds = model.predict(X_val, verbose=0).ravel()
        r2 = r2_score(y_val, preds)

        # Parámetros derivados
        actual_epochs = len(history.history['loss'])
        num_params = model.count_params()

        configs.append({
            "batch_size": batch_size,
            "max_epochs": max_epochs,
            "learning_rate": learning_rate,
            "neurons": neurons,
            "dropout_rate": dropout_rate,
            "num_hidden_layers": num_hidden_layers,
            "activation": activation,
            "actual_epochs": actual_epochs,
            "num_params": num_params
        })
        scores.append(r2)

        # print(f"Model {i+1}/{num_models} trained. R²: {r2:.4f}, batch_size: {batch_size}, max_epochs: {max_epochs}, learning_rate: {learning_rate:.4f}, neurons: {neurons}, dropout_rate: {dropout_rate:.4f}, num_hidden_layers: {num_hidden_layers}, activation: {activation_str}, actual_epochs: {actual_epochs}, num_params: {num_params}")

    # 2. Preparar inputs para AIBA
    evaluations_df = pd.DataFrame(configs)
    evaluations_df["Quality"] = scores
    X_conf = evaluations_df[HYPERPARAMETER_COLUMNS].copy()
    Y_objective = -evaluations_df["Quality"]  # AIBA minimiza; R² se maximiza
    broad_scores = evaluations_df["Quality"].copy()

    # X_conf.to_csv("yacht_model_configs.csv", index=False)

else:
    evaluations_df = pd.read_csv("yacht_model_configs_test.csv")
    X_conf = evaluations_df[HYPERPARAMETER_COLUMNS].copy()
    Y_objective = -evaluations_df["Quality"]  # AIBA minimiza; R² se maximiza
    broad_scores = evaluations_df["Quality"].copy()
    display(evaluations_df.head())


In [ ]:
# X_conf.to_csv("yacht_model_configs_test.csv", index=False)

In [ ]:
display(evaluations_df.sort_values(by="Quality", ascending=False).head(10))
display(evaluations_df.sort_values(by="Quality", ascending=False).tail(10))

num_models = X_conf.shape[0]
print(f"{num_models} modelos cargados.")

In [ ]:
import ast

use_sample_parameters = True  # Cambiar a False para ejecutar AIBA

if not use_sample_parameters:

    # Ejecutar AIBA
    results_df = optimize_intervals(
        X=X_conf,
        Y=Y_objective,
        input_columns=list(X_conf.columns),
        step_int=0.2,
        step_iter=0.01,
        max_steps_int=5,
        max_steps_iter=500,
        M_min=0.01*num_models,
        beta=1
    )

    # Guardar resultados
    results_df.to_csv("results_aiba.csv", index=False)

    print(f"{num_models} modelos generados y AIBA ejecutado. Resultados en results_aiba.csv")

else:
    results_df = pd.read_csv("results_aiba.csv", index_col=False)
    print("Resultados de AIBA cargados desde results_aiba.csv")

    # Columnas que contienen listas
    list_cols = [
        "final_interval_norm",
        "final_interval_denorm",
        "optimal_interval_norm",
        "suboptimal_interval_norm",
        "initial_interval_norm"
    ]

    # Convertir de string a lista real
    for col in list_cols:
        if col in results_df.columns:
            results_df[col] = results_df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)



In [ ]:
# 1. Seleccionar el mejor intervalo según la métrica final
best_row = results_df.sort_values("final_metric").iloc[0]
print("Mejor intervalo encontrado por AIBA (normalizado):")
print("final_interval_denorm", "→", best_row["final_interval_denorm"])
print(f"El valor de la métrica es {best_row['final_metric']}")
print(f"El tamaño del conjunto es {best_row['final_points']}, con un rango de {best_row['final_range']}")

# 2. Muestrear 100 configuraciones dentro de este intervalo
configs_best = []
scores_best = []

intervals_denorm = best_row["final_interval_denorm"]

def sample_log_interval(low, high):
    return float(np.exp(np.random.uniform(np.log(low), np.log(high))))

def sample_uniform_interval(low, high):
    return float(np.random.uniform(low, high))

mins = best_row["final_interval_denorm"][0]
maxs = best_row["final_interval_denorm"][1]

for j in range(100):
    config = {}

    for k, col in enumerate(X_conf.columns):
        low, high = mins[k], maxs[k]
        
        if sampling_type.get(col, "uniform") == "log":
            val = sample_log_interval(max(low, 1e-8), high)  # evitar log(0)
        else:
            val = sample_uniform_interval(low, high)

        if col in ["batch_size", "max_epochs", "neurons", "num_hidden_layers"]:
            val = int(val)

        # caso especial: activation → mapear a string
        if col == "activation":
            activation_val = val
            activation_str = min(activation_mapping.keys(), key=lambda k: abs(activation_mapping[k] - activation_val))
            config["activation_val"] = activation_val
            config["activation_str"] = activation_str
        else:
            config[col] = val

    configs_best.append(config)

X_conf_best = pd.DataFrame(configs_best)

# 3. Entrenar modelos con estas configuraciones (ejemplo con MLP keras)
scores_best = []
for idx, row in X_conf_best.iterrows():
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(input_dim,)))
    for _ in range(int(row["num_hidden_layers"])):
        model.add(keras.layers.Dense(int(row["neurons"]), activation=row["activation_str"]))
        model.add(keras.layers.Dropout(float(row["dropout_rate"])))
    model.add(keras.layers.Dense(output_dim))

    optimizer = keras.optimizers.Adam(learning_rate=row["learning_rate"])
    model.compile(optimizer=optimizer, loss="mse")

    X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=idx)
    
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=int(row["max_epochs"]),
        batch_size=int(row["batch_size"]),
        verbose=0
    )

    preds = model.predict(X_val, verbose=0).ravel()
    score = r2_score(y_val, preds)
    scores_best.append(score)

X_conf_best["Quality"] = scores_best

# 4. Comparación a igual número de modelos: muestra amplia vs. muestra guiada
orig_scores = broad_scores.sample(n=len(scores_best), random_state=0).to_numpy()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(data=[orig_scores, scores_best])
plt.xticks([0,1], ["Muestra amplia (mismo N)", "Muestra guiada por AIBA"])
plt.ylabel("R²")
plt.title("Comparación de rendimiento")
plt.show()

percentil = 75

# 5. Comparar el mismo percentil en dos grupos con igual número de ejecuciones
p10_orig = np.percentile(orig_scores, percentil)
p10_aiba = np.percentile(scores_best, percentil)
print(f"Percentil {percentil}% modelos iniciales: {p10_orig:.4f}")
print(f"Percentil {percentil}% modelos intervalo AIBA: {p10_aiba:.4f}")
print(f"Mejora en percentil {percentil}%: {100*(p10_aiba - p10_orig):.2f}%")